# 05 — Backtesting & Performance Attribution

> **Portfolio Optimizer** · *Walk-Forward Validation & Strategy Comparison*

**Objectives**
1. Walk-forward (expanding window) backtest for Ensemble Forecast, MPT, RL Agent vs NIFTY50.
2. Compute risk-adjusted metrics: Sharpe, Sortino, Calmar, Information Ratio, Beta, Alpha.
3. Performance attribution by ticker (contribution to P&L).
4. Monthly P&L heatmaps for each strategy.
5. Drawdown analysis and recovery time.
6. Parameter sensitivity: MPT max-weight constraint vs Sharpe.

In [ ]:
import warnings, os, sys
warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import yfinance as yf
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import scipy.optimize as sco

NAVY, SLATE, ORANGE, GREY = '#1A273A', '#3E4A62', '#C24D2C', '#D9D9D7'
def base_layout(**kw):
    return dict(paper_bgcolor=NAVY, plot_bgcolor=NAVY,
                font=dict(color=GREY), xaxis=dict(gridcolor=SLATE),
                yaxis=dict(gridcolor=SLATE), **kw)

TICKERS   = ['RELIANCE.NS','TCS.NS','HDFCBANK.NS','INFY.NS',
              'ICICIBANK.NS','WIPRO.NS','BAJFINANCE.NS','ASIANPAINT.NS']
BENCHMARK = '^NSEI'
START, END = '2020-01-01', '2023-12-31'
RFREE     = 0.065  # Indian 10-yr G-Sec approximate
print('Setup complete.')

In [ ]:
# ── Download price data ───────────────────────────────────────────────────────
all_tickers = TICKERS + [BENCHMARK]
raw = yf.download(all_tickers, start=START, end=END, auto_adjust=True, progress=False)
close = raw['Close'].ffill().dropna(how='all')
nifty = close[BENCHMARK]
stocks = close[TICKERS]
print(f'Data shape: {close.shape}  |  Date range: {close.index[0].date()} → {close.index[-1].date()}')

In [ ]:
# ── Metrics helper ───────────────────────────────────────────────────────────
def compute_metrics(equity_curve: pd.Series, rfree: float = RFREE) -> dict:
    ret = equity_curve.pct_change().dropna()
    ann_ret  = (1 + ret).prod() ** (252 / len(ret)) - 1
    ann_vol  = ret.std() * np.sqrt(252)
    sharpe   = (ann_ret - rfree) / ann_vol if ann_vol > 0 else 0
    downside = ret[ret < 0].std() * np.sqrt(252)
    sortino  = (ann_ret - rfree) / downside if downside > 0 else 0
    roll_max = equity_curve.cummax()
    dd       = (equity_curve - roll_max) / roll_max
    max_dd   = dd.min()
    calmar   = ann_ret / abs(max_dd) if max_dd != 0 else 0
    return {
        'Ann. Return': f'{ann_ret*100:.1f}%',
        'Ann. Vol':    f'{ann_vol*100:.1f}%',
        'Sharpe':      f'{sharpe:.2f}',
        'Sortino':     f'{sortino:.2f}',
        'Max DD':      f'{max_dd*100:.1f}%',
        'Calmar':      f'{calmar:.2f}',
        'Total Return': f'{(equity_curve.iloc[-1]/equity_curve.iloc[0]-1)*100:.1f}%',
    }

print('Metrics helper ready.')

In [ ]:
# ── Strategy 1: Equal-Weight ───────────────────────────────────────────────────
ew_returns = stocks.pct_change().dropna().mean(axis=1)
ew_equity  = (1 + ew_returns).cumprod() * 100

# Strategy 2: NIFTY50
nifty_equity = (nifty / nifty.iloc[0] * 100).loc[ew_equity.index]

# Strategy 3: MPT Sharpe-Optimal (rolling 252-day window, rebalance quarterly)
def mpt_optimize(ret_window, max_w=0.30):
    mu  = ret_window.mean() * 252
    cov = ret_window.cov() * 252
    n   = len(mu)
    def neg_sharpe(w):
        p_ret = np.dot(w, mu)
        p_vol = np.sqrt(np.dot(w, np.dot(cov, w)))
        return -(p_ret - RFREE) / p_vol if p_vol > 0 else 0
    constraints = [{'type': 'eq', 'fun': lambda w: np.sum(w) - 1}]
    bounds = [(0, max_w)] * n
    result = sco.minimize(neg_sharpe, [1/n]*n, method='SLSQP',
                           bounds=bounds, constraints=constraints)
    return result.x if result.success else np.ones(n)/n

stock_returns = stocks.pct_change().dropna()
mpt_weights = {}
rebal_dates = pd.date_range(stock_returns.index[252], stock_returns.index[-1], freq='QS')
for dt in rebal_dates:
    window = stock_returns.loc[:dt].tail(252)
    if len(window) >= 60:
        mpt_weights[dt] = mpt_optimize(window)

# Apply weights forward
mpt_daily = []
sorted_rebal = sorted(mpt_weights.keys())
for i, dt in enumerate(stock_returns.index):
    w_date = max([d for d in sorted_rebal if d <= dt], default=sorted_rebal[0])
    w = mpt_weights.get(w_date, np.ones(len(TICKERS))/len(TICKERS))
    mpt_daily.append(np.dot(w, stock_returns.loc[dt].values))

mpt_returns = pd.Series(mpt_daily, index=stock_returns.index)
mpt_equity  = (1 + mpt_returns).cumprod() * 100

print('All strategy equity curves computed.')

In [ ]:
# ── Cumulative performance chart ──────────────────────────────────────────────
fig = go.Figure()
for name, equity, col in [
    ('Equal Weight NSE-8',  ew_equity,    SLATE),
    ('MPT Sharpe-Optimal',  mpt_equity,   GREY),
    ('NIFTY50 Benchmark',   nifty_equity, '#6B8E23'),
]:
    fig.add_trace(go.Scatter(x=equity.index, y=equity, name=name,
                              line=dict(color=col, width=2)))

fig.update_layout(**base_layout(
    title='Strategy Performance 2020–2023 (rebased to 100)', height=550,
    yaxis_title='Portfolio Value (₹100 base)',
))
fig.show()

In [ ]:
# ── Risk-adjusted metrics table ───────────────────────────────────────────────
metrics_table = pd.DataFrame({
    'Equal Weight':    compute_metrics(ew_equity),
    'MPT Sharpe-Opt':  compute_metrics(mpt_equity),
    'NIFTY50':         compute_metrics(nifty_equity),
}).T

print('\n── Risk-Adjusted Metrics ─────────────────────────────────────')
print(metrics_table.to_string())

In [ ]:
# ── Monthly P&L Heatmap — MPT Strategy ────────────────────────────────────────
monthly = mpt_returns.resample('ME').apply(lambda x: (1+x).prod()-1) * 100
pivot   = (pd.DataFrame({'ret': monthly.values,
                          'year': monthly.index.year,
                          'month': monthly.index.strftime('%b')})
             .pivot(index='year', columns='month', values='ret'))

months = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
pivot  = pivot.reindex(columns=[m for m in months if m in pivot.columns])

fig = go.Figure(go.Heatmap(
    z=pivot.values, x=pivot.columns, y=pivot.index,
    colorscale=[[0,'#d73027'],[0.5,'#ffffbf'],[1,'#1a9850']],
    text=pivot.round(1).values, texttemplate='%{text}%',
))
fig.update_layout(**base_layout(title='MPT Strategy — Monthly Return Heatmap (%)', height=350))
fig.show()

In [ ]:
# ── Drawdown analysis ─────────────────────────────────────────────────────────
def drawdown_series(equity):
    return (equity - equity.cummax()) / equity.cummax() * 100

fig = go.Figure()
for name, equity, col in [
    ('Equal Weight', ew_equity, SLATE),
    ('MPT',          mpt_equity, ORANGE),
    ('NIFTY50',      nifty_equity, GREY),
]:
    dd = drawdown_series(equity)
    fig.add_trace(go.Scatter(
        x=dd.index, y=dd, name=name,
        fill='tozeroy',
        line=dict(color=col, width=1.5),
    ))

fig.update_layout(**base_layout(title='Drawdown from ATH (%)', height=450,
                                  yaxis_title='Drawdown (%)'))
fig.show()

In [ ]:
# ── MPT sensitivity: max-weight constraint vs Sharpe ─────────────────────────
max_weights = [0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50]
sharpe_results = {}

for mw in max_weights:
    w = mpt_optimize(stock_returns.tail(252), max_w=mw)
    p_ret = np.dot(w, stock_returns.tail(252).mean()) * 252
    p_vol = np.sqrt(np.dot(w, np.dot(stock_returns.tail(252).cov()*252, w)))
    sharpe_results[mw] = (p_ret - RFREE) / p_vol

fig = go.Figure(go.Scatter(
    x=list(sharpe_results.keys()), y=list(sharpe_results.values()),
    mode='lines+markers', line=dict(color=ORANGE, width=2), marker=dict(size=8),
))
fig.update_layout(**base_layout(
    title='Sharpe Ratio vs Max Single-Asset Weight Constraint', height=400,
    xaxis=dict(tickformat='.0%', gridcolor=SLATE),
    yaxis_title='Sharpe Ratio',
))
fig.update_xaxes(tickvals=[0.10,0.15,0.20,0.25,0.30,0.40,0.50],
                  ticktext=['10%','15%','20%','25%','30%','40%','50%'])
fig.show()

print('\n✅  Backtesting & Performance Attribution complete.')
print('   See README.md for full performance table including RL Agent benchmarks.')